# Full math-on-paper neural network math

In [55]:
import numpy as np

# Suppress scientific notation globally
np.set_printoptions(suppress=True, precision=6)

The network is like this:

Input -> Linear layer + Sigmoid -> Linear layer + Tanh -> Output

![](https://lh3.googleusercontent.com/pw/AP1GczOMNBfoeEIT1EDkHQaWEqGkigcBBbNdZayVPtZwXiuRnJSgzvkCv4uePLvIydyDT8OIC1KyNrEyrj_VLTYB0r5ZHo_83-CW5tI62mRcxcKxGtr7ggV3aUpq3u8XnJbdS61cvb5wbwGaxVxxCo7FvmOW=w1350-h811-s-no-gm?authuser=0)

The sigmoid activation function allows us to stack linear layers preventing them from collapsing into one single linear layer by introducing non-linearity. Sigmoid is usually used inside networks in gating mechanisms and tanh is also used as the output activation when the target output values must fall strictly between -1 and 1, but this is just a mock example to practice, no such mechanisms implied.

Common pitfalls broken down:
- an activation function is NOT a separate node sitting between nodes; it's the back half of the neuron itself.
- the input nodes are NOT neurons, they are just inputs, they don't compute anything, just hold values. A neuron is defined by what it computes: a weighted sum plus bias, then an activation. That's why the nodes in the hidden layer ARE neurons because they summed the values and passed through activation. Each such neuron represents a learnt intermediate feature.
- a number of neurons for the next layer is arbitrary, we follow established architectures usually. The input width is fixed by your data (2 features → 2 inputs) and the final output width is fixed by your task (scalar regression → 1 output). Everything in between — each hidden layer's width and how many hidden layers — is a hyperparameter you pick.

## Step 1. Init

1. Starting point: initialize input $X$ similar to a dataframe for convenience: cols = features, rows = samples.

Then, based on the architecture, the input $X(3x2)$ connects to $3$ neurons in `Hidden Layer 1`. $2$ inputs, each associated with $3$ neurons is $2x3=6$ elements for weights. Given $X(3x2)$, there must be $W(2x3)$ - because the number of cols of $X$ must be equal to the number of rows of $W$:
$$\text{X(3x2) @ w\_input (2x3) = z\_hid\_unbiased(3x3)}$$

To add bias to each of the three rows, we just need to add a row of bias to each: `row_1 + bias_row`, `row_2 + bias_row`, `row_3 + bias_row`:
$$\text{z\_hid\_unbiased(3x3) + b\_hid(1x3) = z\_hid(3x3)}$$

After sigmoid: 
$$\text{h\_hid(3x3) = sigmoid(z\_hid)}$$

2. The next layer - `Hidden Layer 2` - has 3 neurons again, so each of the 3 neurons from `Hidden Layer 1` transitions into each of `Hidden Layer 2`: $3$ neurons into $3$ neurons is $3x3=9$ weights. Then:
$$\text{h\_hid(3x3) @ w\_hidden(3x3) = z\_out\_unbiased(3x3)}$$

And the bias again: 
$$\text{z\_out\_unbiased(3x3) + b\_out(1,3) = z\_out(3x3)}$$

The tanh activation: 
$$\text{h\_out(3x3) = tanh(z\_out)(3x3)}$$

3. The final layer - `Output layer` - has one output exactly, the final prediction. So $3x3$ must be multiplied by something to make exactly $1$ element and having $3$ weights:
$$\text{y\_hat\_unbiased(3x1) = h\_hout(3x3) @ w\_output(3x1)}$$

And the bias again: 
$$\text{y\_hat(3x1) = y\_hat\_unbiased(3x1)+b\_final(1x1)}$$

In [ ]:
# ----- Input/output features -----
# 3 rows, 2 features
X = np.array([
                [2, 3],
                [4, 6],
                [1, -2],
            ])
y = np.array([
                [0.7],
                [5],
                [1],
            ])

# ----- Weights (one weight per feature) -----
# hidden layer 1 weights
w_input = np.array([
                [0.5, 0.1, -0.1],
                [-0.3, 0.2, -0.2],
            ])

# hidden layer 2 weights
w_hidden = np.array([
                [0.1, -0.3, 0.2],
                [0.3, 0.1, 0.1],
                [-0.21, -0.3, -0.1],
            ])

# output layer weights
w_output = np.array([
                [0.1],
                [-0.2],
                [-0.2],
            ])

# ----- Biases -----
# hidden layer 1 bias
b_hid = np.array([
                [0.2, -0.1, 0.3],
            ])
# hidden layer 2 bias
b_out = np.array([
                [0.4, 0.1, 0.05],
            ])

# final layer bias
b_final = np.array([
                [0.1],
            ])

# ----- Learning rate -----
a = 0.1

# ----- Activation functions -----
def sigmoid(x):
    return 1 / (1+np.exp(-x))

def deriv_sigmoid(x):
    return sigmoid(x) * (1 - sigmoid(x))

def tanh(x):
    return ((np.exp(x)) - (np.exp(-x))) / ((np.exp(x)) + (np.exp(-x)))

def deriv_tanh(x):
    return (1 - (tanh(x))**2)

# ----- Layers -----
def linear(features, weights, biases):
    return features @ weights + biases

# ----- Loss function (MSE) -----
def MSE_loss(y_true, y_pred):
    return (1/2)*(y_true - y_pred)**2

def deriv_MSE_loss(y_true, y_pred):
    return y_pred - y_true

## Step 2. Forward pass

In [ ]:
# Hidden layer 1
# (3 samples x 2 features) @ (2 features x 3 weights) + (, 3 biases for each row) = (3 samples x 3 features)
z_hid = linear(X, w_input, b_hid)
h_hid = sigmoid(z_hid)

# Hidden layer 2: tanh(h_hid)
# (3 samples x 3 features) @ (3 features x 3 weights) + (, 3 biases for each row) = (3 samples x 3 features)
z_out = linear(h_hid, w_hidden, b_out)
h_out = tanh(z_out)

# Output layer (linear target)
# (3 samples x 3 features) @ (3 features x 1 weight each) + (1 bias for each row) = (3 samples x 1 target_predicted)
y_hat = linear(h_out, w_output, b_final)
y_hat

array([[0.13749866],
       [0.12503059],
       [0.16344021]])

## Step 3. Compute loss

In [4]:
MSE_loss(y, y_hat)

array([[ 0.15820388],
       [11.8826634 ],
       [ 0.34991614]])

## Step 4. Backpropagation

Ultimately, we want to learn the dependency between the inputs and the outputs, but in a generalized way. We want to understand the data, not memorize the combinations of inputs and outputs. That's why we adjust weights in the way that minimizes the loss. In order to properly adjust the weights, we need to know how weights affect loss: $\frac{\partial L}{\partial W}$.

In a single neuron that derivative was easy. The new problem in a network is that a weight in an early layer touches the loss only through every layer that comes after it — its influence is indirect, buried under several function compositions. This is the exact application of chain rule:

$$\frac{\partial L}{\partial w_{\text{input}}} = \underbrace{\underbrace{\underbrace{\underbrace{\frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial h_{\text{out}}}}_{dL_{h_{\text{out}}}} \cdot \frac{\partial h_{\text{out}}}{\partial z_{\text{out}}}}_{\partial L / \partial z_{\text{out}}} \cdot \frac{\partial z_{\text{out}}}{\partial h_{\text{hid}}}}_{dL_{h_{\text{hid}}}} \cdot \frac{\partial h_{\text{hid}}}{\partial z_{\text{hid}}}}_{\partial L / \partial z_{\text{hid}}} \cdot \frac{\partial z_{\text{hid}}}{\partial w_{\text{input}}}
$$

These all components make our final $\frac{\partial L}{\partial w_{\text{input}}}$, but we don't multiply them like that as we have three sets of weights to update in this example (to see why transpose, look at the full example in code below):

$$\frac{\partial L}{\partial w_{\text{output}}} = h_{\text{out}}^T \cdot \text{dL/d$\hat{y}$} \qquad
\frac{\partial L}{\partial w_{\text{hidden}}} = h_{\text{hid}}^T \cdot \partial L / \partial z_{\text{out}} \qquad
\frac{\partial L}{\partial w_{\text{input}}}  = X^T \cdot \partial L / \partial z_{\text{hid}}$$

Each term calculated (given the loss if MSE):

$$\frac{\partial L}{\partial \hat{y}} = MSE' = \frac{\partial\left[\frac{1}{2}(y-\hat{y})^2\right]}{\partial \hat{y}} = (y-\hat{y}) \cdot (-1) = \hat{y}-y$$

$$\frac{\partial L}{\partial h_{\text{out}}}=\frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial h_{\text{out}}} = (\hat{y}-y) \cdot \frac{\partial (h_{\text{out}} \cdot w_{\text{output}} + b_{\text{final}})}{\partial h_{\text{out}}} = (\hat{y}-y) \cdot w_{\text{output}} \qquad (\text{output is linear, no activation derivative})$$

$$\frac{\partial L}{\partial z_{\text{out}}} = \frac{\partial L}{\partial h_{\text{out}}} \cdot \frac{\partial h_{\text{out}}}{\partial z_{\text{out}}} = w_{\text{w\_output}} \cdot \tanh' = w_{\text{w\_output}} \cdot(1-\tanh(z_{\text{out}})^2)$$

$$\frac{\partial L}{\partial h_{\text{hid}}} = \frac{\partial L}{\partial z_{\text{out}}} \cdot \frac{\partial z_{\text{out}}}{\partial h_{\text{hid}}} = \frac{\partial L}{\partial z_{\text{out}}} \cdot\frac{\partial (h_{\text{hid}} \cdot w_{\text{hidden}} + b_{\text{out}})}{\partial h_{\text{hid}}} = \frac{\partial L}{\partial z_{\text{out}}} \cdot w_{\text{hidden}}$$

$$\frac{\partial L}{\partial z_{\text{hid}}} = \frac{\partial L}{\partial h_{\text{hid}}} \cdot \frac{\partial h_{\text{hid}}}{\partial z_{\text{hid}}} = \frac{\partial L}{\partial h_{\text{hid}}} \cdot \sigma' = \frac{\partial L}{\partial h_{\text{hid}}} \cdot\sigma(z_{\text{hid}}) \cdot (1-\sigma(z_{\text{hid}}))$$


Now, it's important to note that every single value of each matrix is a scalar and chain rule must be applied to each of them. So for each $w_i$ in weights, we do

$$\text{h\_out}_0*\text{w\_output}_0 + \text{h\_out}_1 * \text{w\_output}_1 + \text{h\_out}_2 * \text{w\_output}_2 \leftarrow \text{and this is a dot product exactly (@)!}$$

In [ ]:
# -------- output layer --------
dL_dy_hat    = deriv_MSE_loss(y, y_hat)                     # (3,1)
# here we have # (3 samples x 3 features) @ (3 samples x 1 predicted_target) - axis misalignment, the inner axis must be the same, transpose: (3 features x 3 samples) @ (3 samples x 1 pred_target)
dL_dw_output = h_out.T @ dL_dy_hat                          # (3,1)

# -------- hidden layer 2 (tanh) --------
# (3 samples x 1 pred_target) @ (3 features x 1 weight each) - axis misalignment, transpose again: (3 samples x 1 pred_target) @ (1 weight each x 3 features)
dL_dh_out    = dL_dy_hat @ w_output.T                       # (3,3)
dL_dz_out   = dL_dh_out * deriv_tanh(z_out)                 # scalar multiplication to account for activation derivative element-wise (3 samples x 3 neurons)
# (3 samples x 3 features) @ (3 samples x 3 features) - axis misalignment, transpose again: (3 features x 3 samples) @ (3 samples x 3 features)
dL_dw_hidden = h_hid.T @ dL_dz_out                          # (3 features x 3 features)

# -------- hidden layer 1 (sigmoid) --------
# (3 samples x 3 neurons) @ (3 features x 3 weights) - axis misalignment, transpose again: (3 samples x 3 neurons) @ (3 weights x 3 features)
dL_dh_hid   = dL_dz_out @ w_hidden.T                # (3 samples, 3 features)  like h_hid
dL_dz_hid   = dL_dh_hid * deriv_sigmoid(z_hid)      # scalar multiplication to account for activation derivative element-wise (3 samples x 3 neurons)  like z_hid
# (3 samples x 2 features) @ (3 samples x 3 neurons) - axis misalignment, transpose again: (2 features x 3 samples) @ (3 samples x 3 neurons)
dL_dw_input = X.T @ dL_dz_hid                       # (2 feat, 3 neurons)

## Step 5. Update (gradient descent step)

In [52]:
w_output -= a * dL_dw_output
w_hidden -= a * dL_dw_hidden
w_input -= a * dL_dw_input

# biases: sum each dL_dz over the samples axis
dL_db_final = dL_dy_hat.sum(axis=0, keepdims=True)   # (1,1)
dL_db_out   = dL_dz_out.sum(axis=0, keepdims=True)   # (1,3)
dL_db_hid   = dL_dz_hid.sum(axis=0, keepdims=True)   # (1,3)

b_final  -= a * dL_db_final
b_out    -= a * dL_db_out
b_hid    -= a * dL_db_hid

## Step 6. Repeat steps 2-5 until the loss stops decreasing

In [53]:
# Hidden layer 1
z_hid = linear(X, w_input, b_hid)
h_hid = sigmoid(z_hid)

# Hidden layer 2: tanh(h_hid)
z_out = linear(h_hid, w_hidden, b_out)
h_out = tanh(z_out)

# Output layer (linear target)
y_hat = linear(h_out, w_output, b_final)
y_hat

array([[1.06139009],
       [1.07377651],
       [1.06166232]])

In [56]:
MSE_loss(y, y_hat)

array([[0.065301],
       [7.707615],
       [0.001901]])